# 04 — SHAP Point Explanations (Observation-Level)

Produces signed SHAP values at the **individual row level** for every run
defined in `manifesto_config.pkl`.

| Column | Content |
|---|---|
| Identity columns | `Segment`, `Cust_ID`, `WeekYear`, `UnitSales`, `pred_col` |
| `shap_<feature>` | Signed SHAP contribution of that feature for this row |

**Sampling note for Neural (LSTM)**  
GradientExplainer runs on all rows in batch mode — no sampling needed.
The flat-sequence approximation is the same as in `03_shap_explanations.ipynb`.

**Outputs** (one file per run):  
`Outputs/shap_local/{model_key}_shap_{ds_key}.csv`

**No inputs to change.** All configuration is driven by `manifest.csv` via `00_input_manifesto.ipynb`.

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.shap_utils import load_model_artifacts, compute_shap

pd.set_option('display.max_columns', 50)

NOTEBOOK_DIR = os.path.abspath('')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
LOCAL_DIR    = os.path.join(OUTPUT_DIR, 'shap_local')
os.makedirs(LOCAL_DIR, exist_ok=True)

# Identity columns to carry through to output
ID_COLS = ['Segment', 'Cust_ID', 'WeekYear', 'Year', 'WeekNum']

print('Libraries loaded')

## 1. Load Manifesto

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'manifesto_config.pkl'), 'rb') as f:
    manifesto = pickle.load(f)

dataset_configs = manifesto['dataset_configs']
dev_lookup      = {r['model']: r for r in dataset_configs if r['dataset'] == 'DEV'}

print(f'Runs scheduled : {len(dataset_configs)}')

## 2. Point SHAP Pipeline — All Runs

In [ ]:
artifacts_cache  = {}   # model_name -> (model_obj, features, aux)
background_cache = {}   # model_name -> background_df

for run in dataset_configs:
    run_id       = run['run_id']
    model_name   = run['model']
    model_family = run['model_family']
    model_type   = run['model_type']

    print(f'\n[{run_id}]  family={model_family}  type={model_type}')

    # ── Load prediction file (from manifest path) ──────────────────────────────
    df = pd.read_csv(run['path'])

    # ── Background ────────────────────────────────────────────────────────────
    if model_name not in background_cache:
        dev_run = dev_lookup.get(model_name)
        background_cache[model_name] = pd.read_csv(dev_run['path']) if dev_run else df
    background_df = background_cache[model_name]

    # ── Load model artifacts (cached) ─────────────────────────────────────────
    if model_name not in artifacts_cache:
        print(f'  Loading model artifacts ...')
        model_obj, features, aux = load_model_artifacts(run)
        artifacts_cache[model_name] = (model_obj, features, aux)
        print(f'  Features : {len(features)}')
    else:
        model_obj, features, aux = artifacts_cache[model_name]

    # ── Compute SHAP (all rows) ────────────────────────────────────────────────
    shap_arr, eff_features = compute_shap(
        df, features, model_obj, model_family, aux, background_df,
        model_type=model_type,
    )
    print(f'  SHAP array : {shap_arr.shape}')

    # ── Build wide output ─────────────────────────────────────────────────────
    df_reset = df.reset_index(drop=True)
    id_available = [c for c in ID_COLS + [run['target_col'], run['pred_col']]
                    if c in df_reset.columns]
    out = df_reset[id_available].copy()

    for j, feat in enumerate(eff_features):
        out[f'shap_{feat}'] = shap_arr[:, j].round(6)

    # ── Save ──────────────────────────────────────────────────────────────────
    model_key = model_name.lower().replace(' ', '_')
    ds_key    = run['dataset'].lower()
    file_path = os.path.join(LOCAL_DIR, f'{model_key}_shap_{ds_key}.csv')
    out.to_csv(file_path, index=False)
    print(f'  Saved: {os.path.basename(file_path)}  ({len(out):,} rows  {len(eff_features)} features)')

print('\nPoint SHAP pipeline complete.')

## 3. Quick Sample View

In [ ]:
# Preview the last file written
import glob
files = sorted(glob.glob(os.path.join(LOCAL_DIR, '*.csv')))
if files:
    preview = pd.read_csv(files[0])
    shap_cols = [c for c in preview.columns if c.startswith('shap_')]
    print(f'File : {os.path.basename(files[0])}')
    print(f'Rows : {len(preview):,}  |  SHAP features : {len(shap_cols)}')
    display(preview.head(5))